# ML_G1G2_P02toP07_DEEP_REENTRY

## Title / 목적

이 노트북은 Gate 1 후반부터 Gate 2 전체를 한 번에 복구하기 위한 심층 재진입 문제지다. 범위는 Perceptron·XOR·loss/gradient·chain rule에서 시작해 2-2-1 MLP backprop 손계산, Layer abstraction, mini-batch vectorization, Network abstraction까지다.

- 정답 포함: 아니오
- 코드 모드: `hint_only`
- 총점: 100점
- 통과 기준: 85점 이상 G1/G2 통과
- 주의: `ML_G1G2_P02toP07_DEEP_REENTRY_answer.ipynb`는 풀이 후 열 것


## Source Map

| node_id | 강의 | 이 문제 세트에서 쓰는 역할 |
|---|---:|---|
| `n_ML8.perceptron` | 8강 | Perceptron = weighted sum + activation, `z = w^T x + b` |
| `n_ML8.forward_loss_backward_update` | 8강 | Forward / Loss / Backward / Update 학습 루프 |
| `n_ML8.xor` | 8강 | XOR는 단일 Perceptron으로 선형 분리 불가능 |
| `n_ML8.mlp_xor` | 8강 | hidden layer와 nonlinear activation으로 XOR 해결 |
| `n_ML10.chain_rule` | 10강 | 합성함수의 local derivative 곱 구조 |
| `n_ML10.gradient_descent` | 10강 | gradient는 loss 증가 방향, GD는 반대 방향 |
| `n_ML10.backprop` | 10강 | 계산 그래프 전체에 chain rule 반복 적용 |
| `n_ML11.section_02` | 11강 | 2-2-1 MLP 구조와 파라미터 shape |
| `n_ML11.forward_pass` | 11강 | `z1, a1, z2, a2` forward cache |
| `n_ML11.section_05` | 11강 | output delta와 출력층 gradient |
| `n_ML11.section_06` | 11강 | hidden delta, `W2.T`, `@`와 `*` 구분 |
| `n_ML11.forward_vs_backward` | 11강 | forward 값과 backward gradient의 역할 구분 |
| `n_ML12.backprop` | 12강 | 2-2-1 backprop 패턴을 일반 layer로 확장 |
| `n_ML12.layer_abstraction_activation` | 12강 | `Layer`, `Dense`, `ReLU`, `Sigmoid`, `SoftmaxCE` |
| `n_ML12.vectorized_backprop_batch_dimension_matrix` | 12강 | batch dimension과 Dense backward shape 관계 |
| `n_ML12.network_mini_batch` | 12강 | `Network.forward`, reversed `backward`, mini-batch `fit` |
| `n_ML13.keras` | 13강 | 직접 구현 `Network`와 Keras `Sequential/compile/fit` 대응 |


## Gate Map

| 범위 | 노드 | 이번 노트북에서 확인할 것 |
|---|---|---|
| Gate 1 | G1-P02 MLP와 XOR | Perceptron 한계, hidden layer와 nonlinear activation의 필요성 |
| Gate 1 | G1-P03 Loss와 Gradient | loss 최소화, gradient 방향, optimizer 책임 |
| Gate 1 | G1-P04 Chain Rule | local derivative 곱, backprop 언어 번역 |
| Gate 1 | G1-P05 Forward / Loss / Backward / Update | cache와 gradient/update의 역할 분리 |
| Gate 2 | G2-P01 2-2-1 MLP forward | `z1, a1, z2, a2, loss` 손계산 |
| Gate 2 | G2-P02 Output delta | `delta2 = a2 - y`, `dW2`, `db2` |
| Gate 2 | G2-P03 Hidden delta | `W2.T @ delta2`, sigmoid local derivative |
| Gate 2 | G2-P04 Dense.backward | `dW`, `db`, `dX` 공식과 shape |
| Gate 2 | G2-P05 Activation / SoftmaxCE | parameter 없는 layer, `(p-y)/B` |
| Gate 2 | G2-P06 Mini-batch vectorization | batch axis 0, outer product의 행렬곱화 |
| Gate 2 | G2-P07 Network abstraction | layer list, forward 순방향, backward 역방향 |
| 연결 | Gate 3 제외, Keras 연결만 | `Sequential`, `compile`, `fit`, optimizer 책임 확인 |


## 데이터 자산 확인

풀이에 필요한 소형 데이터는 `data/ML_G1G2_P02toP07_DEEP_REENTRY/` 아래에 있다.

| 파일 | 용도 |
|---|---|
| `xor_data.csv` | XOR 진리표와 선형 분리 판단 |
| `mlp_221_fixed.npz` | 2-2-1 MLP 손계산용 `x, y, W1, b1, W2, b2` |
| `mlp_221_fixed.json` | 같은 파라미터의 사람이 읽기 쉬운 버전 |
| `minibatch_shape_examples.npz` | Dense backward와 mini-batch shape 추적 |
| `binary_classification_toy.csv` | 최종 binary classification 시나리오 |
| `iris_like_3class_toy.csv` | SoftmaxCE와 multiclass 비교용 toy data |
| `metadata.json` | 데이터 설명 |


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("data/ML_G1G2_P02toP07_DEEP_REENTRY")
required_files = [
    "xor_data.csv",
    "mlp_221_fixed.npz",
    "mlp_221_fixed.json",
    "minibatch_shape_examples.npz",
    "binary_classification_toy.csv",
    "iris_like_3class_toy.csv",
    "metadata.json",
]

assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"
for name in required_files:
    assert (DATA_DIR / name).exists(), f"Missing data file: {name}"

print("DATA_DIR ready:", DATA_DIR)
print("files:", sorted(p.name for p in DATA_DIR.iterdir()))

# 힌트:
# xor = pd.read_csv(DATA_DIR / "xor_data.csv")
# params = np.load(DATA_DIR / "mlp_221_fixed.npz")
# batch = np.load(DATA_DIR / "minibatch_shape_examples.npz")
# binary = pd.read_csv(DATA_DIR / "binary_classification_toy.csv")


## 채점 기준

총점 100점.

| 평가 축 | 배점 |
|---|---:|
| 개념 연결 | 20점 |
| 수식 / chain rule | 20점 |
| 2-2-1 손계산 | 25점 |
| shape / vectorization | 20점 |
| layer / Keras 연결 | 10점 |
| 설명 명료성 | 5점 |

통과 기준:

| 점수 | 판정 |
|---:|---|
| 85점 이상 | G1/G2 통과 |
| 70~84점 | 계산은 가능하나 shape/backprop 언어 보강 필요 |
| 50~69점 | chain rule과 delta 재학습 필요 |
| 50점 미만 | G1-P02부터 재시작 |


## 문제 1. Perceptron과 MLP [Q01-A, 5점]

**Source nodes:** `n_ML8.perceptron`, `n_ML8.mlp_xor`

### 문제 원문

1. Perceptron의 계산식을 `z = ...`, `a = ...` 형태로 쓰시오.
2. activation function이 weighted sum 뒤에 붙는 이유를 설명하시오.
3. MLP가 Perceptron 여러 개를 어떤 방식으로 조합한 구조인지 설명하시오.
4. 단순히 layer를 많이 쌓는 것과 nonlinear activation을 포함해 쌓는 것의 차이를 쓰시오.


### 채점 기준
계산식 1점, activation 역할 1.5점, MLP 구조 1.5점, 비선형성 필요성 1점

### 유도 질문 / 자주 틀리는 지점
activation을 단순 threshold나 장식으로만 설명하지 말 것.


In [ ]:
# 힌트:
# z, w, x, b, activation이라는 단어를 모두 사용한다.
# MLP 설명에는 hidden layer와 nonlinear representation을 포함한다.
# 완성 문장은 직접 작성한다.


### 내 답안


- 계산식:
- activation의 역할:
- MLP 구조 설명:
- 내가 헷갈린 지점:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 2. XOR와 hidden layer의 필요성 [Q02-B, 5점]

**Source nodes:** `n_ML8.xor`, `n_ML8.mlp_xor`

### 문제 원문

AND/OR/NAND는 단일 Perceptron으로 가능하지만 XOR는 왜 불가능한지 설명하시오.

다음 두 관점을 모두 포함하시오.
1. 원래 입력 공간에서의 선형 분리 관점
2. hidden layer가 입력 공간을 변환한다는 관점

마지막에 “MLP가 XOR를 해결한다”는 문장을 논리 연산 조합 또는 representation 변환 중 하나로 설명하시오.


### 채점 기준
선형 분리 2점, hidden 변환 2점, MLP 연결 1점

### 유도 질문 / 자주 틀리는 지점
“XOR는 복잡해서”가 아니라 “대각선 배치라 직선 하나로 분리 불가”라고 설명할 것.


In [ ]:
# 힌트:
# xor = pd.read_csv(DATA_DIR / "xor_data.csv")
# 산점도를 직접 그려 대각선 배치를 확인할 수 있다.
# 단, 정답 문장은 그림 결과를 근거로 직접 작성한다.


### 내 답안


- 선형 분리 불가능한 이유:
- hidden layer가 하는 일:
- MLP 필요성:
- 유도 질문 답: XOR는 직선 하나로 나뉘는가?


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 3. Loss, Gradient, Optimizer 책임 [Q03-C, 5점]

**Source nodes:** `n_ML10.gradient_descent`, `n_ML13.keras`

### 문제 원문

다음 문장을 판정하고, 틀린 문장은 고치시오.

1. loss는 예측과 정답의 차이를 하나의 숫자로 만든다.
2. gradient는 loss가 감소하는 방향이다.
3. gradient descent는 gradient와 같은 방향으로 움직인다.
4. optimizer는 gradient를 새로 유도하는 존재가 아니라, 계산된 gradient로 parameter를 움직이는 update 규칙이다.

각 문장마다 O/X와 근거를 쓰시오.


### 채점 기준
loss 정의 1점, gradient 방향 1.5점, GD update 방향 1.5점, optimizer 책임 1점

### 유도 질문 / 자주 틀리는 지점
optimizer를 backprop/gradient 계산기와 동일시하지 말 것.


In [ ]:
# 힌트:
# gradient 방향과 update 방향을 화살표로 구분해 본다.
# theta_new = theta - lr * grad 형태를 떠올린다.
# optimizer의 책임은 gradient 계산인지 parameter 이동인지 구분한다.


### 내 답안


| 번호 | O/X | 근거 | 고친 문장 |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 4. Chain Rule을 backprop 언어로 번역하기 [Q04-D, 5점]

**Source nodes:** `n_ML10.chain_rule`, `n_ML10.backprop`

### 문제 원문

합성함수 `y = f(g(x))`에서 `dy/dx`가 어떤 local derivative들의 곱으로 표현되는지 쓰시오.

그 다음, 신경망 backprop 언어로 다음 문장을 완성하시오.

> 어떤 layer의 backward는 “위에서 온 gradient × ________”를 계산해서 아래 layer로 넘긴다.

또한 경로가 여러 개일 때 gradient 기여가 어떻게 합쳐지는지 한 문장으로 설명하시오.


### 채점 기준
chain rule 수식 2점, upstream/local 표현 2점, 여러 경로 합산 1점

### 유도 질문 / 자주 틀리는 지점
local derivative 하나만 쓰고 위에서 온 gradient를 빼먹지 말 것.


In [ ]:
# 힌트:
# y=f(u), u=g(x)로 중간 변수를 둔다.
# local derivative, upstream gradient, sum over paths를 사용한다.


### 내 답안


- 합성함수 미분:
- backprop 번역 문장:
- 여러 경로가 있을 때:
- 유도 질문 답: “위에서 온 gradient”는 어느 방향에서 오는가?


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 5. Forward / Loss / Backward / Update와 cache [Q05-E, 5점]

**Source nodes:** `n_ML8.forward_loss_backward_update`, `n_ML11.forward_vs_backward`

### 문제 원문

네 단계를 올바른 순서로 쓰고, 각 단계의 산출물과 cache 여부를 구분하시오.

| 단계 | 무엇을 계산하는가 | 다음 단계에 넘기는 값 | cache에 저장해야 하는 값 |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |

특히 `z1, a1, z2, a2` 중 backward에서 필요한 값을 표시하시오.


### 채점 기준
순서 1점, 단계별 값 2점, cache 구분 1점, update/grads 구분 1점

### 유도 질문 / 자주 틀리는 지점
update 단계에서 gradient를 계산한다고 쓰지 말 것.


In [ ]:
# 힌트:
# forward는 예측과 cache를 만든다.
# loss는 scalar loss와 시작 gradient의 근거를 만든다.
# backward는 grads를 만든다.
# update는 grads를 사용해 parameter를 움직인다.


### 내 답안


| 단계 | 계산 | 전달값 | cache |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 6. 2-2-1 MLP 구조와 shape 표 [Q06-F0, 6점]

**Source nodes:** `n_ML11.section_02`, `n_ML11.forward_pass`

### 문제 원문

다음 구조의 shape를 모두 채우시오.

- input 2 → hidden 2 sigmoid → output 1 linear
- 단일 샘플 convention: `x: (2,)`, `W1 @ x + b1`
- `W1 = [[0.1, 0.2], [0.3, 0.4]]`, `b1 = [0,0]`
- `W2 = [[0.5, 0.6]]`, `b2 = [0]`

| 기호 | 의미 | shape | forward/backward 중 쓰임 |
|---|---|---:|---|
| x |  |  |  |
| W1 |  |  |  |
| b1 |  |  |  |
| z1 |  |  |  |
| a1 |  |  |  |
| W2 |  |  |  |
| b2 |  |  |  |
| z2/a2 |  |  |  |
| loss |  |  |  |


### 채점 기준
파라미터 shape 2점, cache shape 2점, forward/backward 쓰임 1점, convention 일관성 1점

### 유도 질문 / 자주 틀리는 지점
`W1`의 행을 input feature로 착각하지 말 것. 행은 hidden neuron이다.


In [ ]:
# 힌트:
# params = np.load(DATA_DIR / "mlp_221_fixed.npz")
# print(params["W1"].shape)처럼 shape만 확인할 수 있다.
# 계산값은 직접 손으로 채운다.


### 내 답안


| 기호 | 의미 | shape | 근거 |
|---|---|---:|---|
| x |  |  |  |
| W1 |  |  |  |
| b1 |  |  |  |
| z1 |  |  |  |
| a1 |  |  |  |
| W2 |  |  |  |
| b2 |  |  |  |
| z2/a2 |  |  |  |
| loss |  |  |  |


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 7. 2-2-1 MLP forward 손계산 [Q07-F, 7점]

**Source nodes:** `n_ML11.forward_pass`

### 문제 원문

고정값으로 forward를 손계산하시오.

- `x = [1, 1]`, `y = 0`
- `W1 = [[0.1,0.2],[0.3,0.4]]`, `b1 = [0,0]`
- `W2 = [[0.5,0.6]]`, `b2 = [0]`
- hidden activation: sigmoid
- output activation: linear
- loss: `1/2 * (y - a2)^2`

| 값 | 계산식 | 근삿값 |
|---|---|---:|
| z1[0] |  |  |
| z1[1] |  |  |
| a1[0] |  |  |
| a1[1] |  |  |
| z2 |  |  |
| a2 |  |  |
| loss |  |  |

허용 오차: 소수점 넷째 자리 기준 ±0.001.


### 채점 기준
z1 1.5점, a1 2점, z2/a2 2점, loss 1점, 오차 범위·표기 0.5점

### 유도 질문 / 자주 틀리는 지점
출력층에 sigmoid를 다시 적용하지 말 것.


In [ ]:
# 힌트:
# sigmoid는 직접 정의해도 된다.
# def sigmoid(t):
#     return 1 / (1 + np.exp(-t))
# z1, a1, z2, a2, loss 순서로 계산한다.
# 문제지에는 완성 코드를 쓰지 말고, 손계산 표를 채운다.


### 내 답안


| 값 | 계산식 | 근삿값 |
|---|---|---:|
| z1[0] |  |  |
| z1[1] |  |  |
| a1[0] |  |  |
| a1[1] |  |  |
| z2 |  |  |
| a2 |  |  |
| loss |  |  |


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 8. 계산 그래프와 local derivative 경로 [Q08-Bridge, 7점]

**Source nodes:** `n_ML10.chain_rule`, `n_ML11.forward_vs_backward`

### 문제 원문

2-2-1 MLP에서 다음 경로를 chain rule 관점으로 채우시오.

1. `W2[0,0] → z2 → a2 → loss`
2. `W1[0,0] → z1[0] → a1[0] → z2 → a2 → loss`

| 대상 parameter | 경로 | 곱해지는 local derivative | 짧은/긴 체인 |
|---|---|---|---|
| W2[0,0] |  |  |  |
| W1[0,0] |  |  |  |

질문: 왜 은닉층 parameter는 출력층 parameter보다 chain이 긴가?


### 채점 기준
출력층 경로 2점, 은닉층 경로 3점, local derivative 언어 1점, 긴 체인 이유 1점

### 유도 질문 / 자주 틀리는 지점
은닉층에서 sigmoid 미분을 빼먹지 말 것.


In [ ]:
# 힌트:
# 출력층 weight는 a1과 delta2만 연결하면 된다.
# 은닉층 weight는 sigmoid local derivative와 W2를 거친다.
# 수치 계산보다 경로 언어를 정확히 쓰는 문제다.


### 내 답안


| 대상 parameter | 경로 | local derivative | 짧은/긴 체인 |
|---|---|---|---|
| W2[0,0] |  |  |  |
| W1[0,0] |  |  |  |
- 은닉층 chain이 긴 이유:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 9. Output delta와 출력층 gradient [Q09-G, 7점]

**Source nodes:** `n_ML11.section_05`

### 문제 원문

문항 7의 forward 값을 사용한다.

1. `L = 1/2(y-a2)^2`를 `a2`로 미분하여 `delta2 = a2 - y`가 나오는 이유를 설명하시오.
2. `dW2 = delta2 * a1`, `db2 = delta2`를 계산하시오.
3. `dW2`와 `db2`의 shape를 쓰시오.

| 값 | 계산식 | 근삿값 | shape |
|---|---|---:|---:|
| delta2 |  |  |  |
| dW2[0,0] |  |  |  |
| dW2[0,1] |  |  |  |
| db2 |  |  |  |


### 채점 기준
delta 유도 2점, dW2 수치 2점, db2와 bias 이유 2점, shape 1점

### 유도 질문 / 자주 틀리는 지점
`y-a2`와 `a2-y` 부호를 바꾸지 말 것.


In [ ]:
# 힌트:
# 반쪽 MSE라서 2가 사라진다.
# dW2는 np.outer(delta2, a1) 형태로 생각하면 shape가 안정적이다.
# bias는 z2에 +1 계수로 더해진다.


### 내 답안


- delta2 유도:
| 값 | 계산식 | 근삿값 | shape |
|---|---|---:|---:|
| delta2 |  |  |  |
| dW2[0,0] |  |  |  |
| dW2[0,1] |  |  |  |
| db2 |  |  |  |
- bias gradient가 delta와 같은 이유:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 10. Hidden delta, W2.T, @와 * [Q10-H, 7점]

**Source nodes:** `n_ML11.section_06`

### 문제 원문

문항 7~9의 값을 사용한다.

1. `dL/da1 = W2.T @ delta2`를 계산하시오.
2. `delta1 = dL/da1 * a1 * (1-a1)`를 계산하시오.
3. 왜 `W2`가 아니라 `W2.T`가 필요한지 설명하시오.
4. 이때 `@`와 `*`의 차이를 설명하시오.

| 값 | 계산식 | 근삿값 | shape |
|---|---|---:|---:|
| dL/da1[0] |  |  |  |
| dL/da1[1] |  |  |  |
| delta1[0] |  |  |  |
| delta1[1] |  |  |  |


### 채점 기준
dL/da1 2점, delta1 2점, W2.T 이유 1.5점, @/* 구분 1.5점

### 유도 질문 / 자주 틀리는 지점
`delta1 = dL_da1 @ a1 * ...`처럼 elementwise 위치에 행렬곱을 넣지 말 것.


In [ ]:
# 힌트:
# W2 shape는 (1,2), W2.T shape는 (2,1)이다.
# @는 선형 결합/행렬곱, *는 같은 위치끼리 곱하는 elementwise 곱이다.
# sigmoid derivative는 a1 * (1-a1)을 사용한다.


### 내 답안


| 값 | 계산식 | 근삿값 | shape |
|---|---|---:|---:|
| dL/da1[0] |  |  |  |
| dL/da1[1] |  |  |  |
| delta1[0] |  |  |  |
| delta1[1] |  |  |  |
- W2.T가 필요한 이유:
- @와 *의 차이:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 11. Hidden dW/db와 shape [Q11-I, 6점]

**Source nodes:** `n_ML11.section_06`, `n_ML11.forward_vs_backward`

### 문제 원문

문항 10의 `delta1`을 사용해 hidden layer gradient를 계산하시오.

- `dW1 = outer(delta1, x)`
- `db1 = delta1`

| 값 | 계산식 | 근삿값 | shape |
|---|---|---:|---:|
| dW1[0,0] |  |  |  |
| dW1[0,1] |  |  |  |
| dW1[1,0] |  |  |  |
| dW1[1,1] |  |  |  |
| db1[0] |  |  |  |
| db1[1] |  |  |  |

질문: `x=[1,1]`이어서 수치가 같아지는 항이 있더라도 shape를 생략하면 안 되는 이유를 설명하시오.


### 채점 기준
dW1 2점, db1 1.5점, shape 1.5점, 일반화 설명 1점

### 유도 질문 / 자주 틀리는 지점
`dW1`을 `(2,)`로 줄여 쓰지 말 것.


In [ ]:
# 힌트:
# np.outer(delta1, x)의 결과 shape를 먼저 생각한다.
# bias gradient는 z1에 대한 delta와 같은 shape다.
# x 값이 모두 1인 것은 우연한 예제 조건이다.


### 내 답안


| 값 | 계산식 | 근삿값 | shape |
|---|---|---:|---:|
| dW1[0,0] |  |  |  |
| dW1[0,1] |  |  |  |
| dW1[1,0] |  |  |  |
| dW1[1,1] |  |  |  |
| db1[0] |  |  |  |
| db1[1] |  |  |  |
- shape를 생략하면 안 되는 이유:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 12. 2-2-1 backward 전체와 gradient check 의미 [Q12-BackpropSummary, 5점]

**Source nodes:** `n_ML11.forward_vs_backward`, `n_ML11.gradient_check`

### 문제 원문

문항 9~11을 하나의 backward 결과로 묶으시오.

1. `grads = {dW1, db1, dW2, db2}`의 shape를 쓰시오.
2. 한 번의 update 식을 `W <- W - lr * dW` 형태로 쓰시오.
3. gradient check가 무엇을 확인하는지 설명하시오.
4. gradient check가 실패했을 때 가장 먼저 의심할 지점을 두 가지 쓰시오.


### 채점 기준
grads shape 1.5점, update 식 1점, gradient check 의미 1.5점, 실패 원인 1점

### 유도 질문 / 자주 틀리는 지점
gradient check를 “학습 성능 평가”라고 쓰지 말 것.


In [ ]:
# 힌트:
# gradient check는 수치미분과 backward가 만든 해석적 gradient를 비교한다.
# 실패 시 부호, transpose, @와 *, loss scalar화부터 본다.


### 내 답안


| gradient | shape | update 대상 |
|---|---:|---|
| dW1 |  |  |
| db1 |  |  |
| dW2 |  |  |
| db2 |  |  |
- update 식:
- gradient check 의미:
- 실패 시 의심 지점:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 13. Dense.backward와 Activation backward 코드 빈칸 [Q13-JK, 7점]

**Source nodes:** `n_ML12.layer_abstraction_activation`, `n_ML12.vectorized_backprop_batch_dimension_matrix`

### 문제 원문

batch convention은 `X: (B,D)`, `W: (H,D)`, `b: (H,)`이고 forward는 `Z = X @ W.T + b`이다. upstream gradient `dZ`의 shape는 `(B,H)`이다.

아래 빈칸을 채우고 shape를 설명하시오.

```python
class Dense:
    def backward(self, dZ):
        self.dW = __________
        self.db = __________
        dX = __________
        return dX
```

또한 ReLU와 Sigmoid backward를 비교하시오.

| layer | parameter 있음? | local derivative | update 대상인가? |
|---|---|---|---|
| Dense |  |  |  |
| ReLU |  |  |  |
| Sigmoid |  |  |  |


### 채점 기준
Dense 공식 3점, shape 2점, activation 비교 1.5점, update 대상 구분 0.5점

### 유도 질문 / 자주 틀리는 지점
`dW = X.T @ dZ`는 convention이 바뀐 식이다. 여기서는 W가 `(H,D)`임을 확인할 것.


In [ ]:
# 힌트:
# dW는 W와 같은 shape여야 한다.
# db는 batch 축(axis=0)으로 더한다.
# dX는 앞 layer로 넘길 gradient다.
# ReLU/Sigmoid는 update할 W,b가 없다.


### 내 답안


```python
self.dW = 
self.db = 
dX = 
```
| 값 | shape | 이유 |
|---|---:|---|
| dW |  |  |
| db |  |  |
| dX |  |  |
| ReLU backward |  |  |
| Sigmoid backward |  |  |


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 14. SoftmaxCE 함수 카드 [Q14-L, 6점]

**Source nodes:** `n_ML12.layer_abstraction_activation`, `n_ML12.vectorized_backprop_batch_dimension_matrix`

### 문제 원문

SoftmaxCE를 함수 카드처럼 정리하시오.

1. 입력 logits의 shape와 의미
2. softmax가 만드는 probs의 shape와 의미
3. cross entropy loss의 의미
4. backward에서 `p-y` 또는 `(p-y)/B`가 나오는 이유
5. binary sigmoid와 multiclass softmax의 차이

| 항목 | binary sigmoid | multiclass softmax |
|---|---|---|
| output 수 |  |  |
| target 표현 |  |  |
| loss |  |  |
| backward 출발점 |  |  |


### 채점 기준
logits/probs/loss 2점, backward 2점, binary vs multiclass 2점

### 유도 질문 / 자주 틀리는 지점
softmax output을 독립적인 여러 sigmoid로 착각하지 말 것.


In [ ]:
# 힌트:
# logits는 아직 확률이 아니다.
# probs는 class 축으로 합이 1이다.
# batch 평균 loss라면 backward에서 /B가 들어간다.
# binary와 multiclass는 output layer 설계가 다르다.


### 내 답안


- logits:
- probs:
- CE loss:
- backward 출발점:
| 항목 | binary sigmoid | multiclass softmax |
|---|---|---|
| output 수 |  |  |
| target 표현 |  |  |
| loss |  |  |
| backward 출발점 |  |  |


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 15. Mini-batch vectorization과 Network abstraction [Q15-MN, 7점]

**Source nodes:** `n_ML12.vectorized_backprop_batch_dimension_matrix`, `n_ML12.network_mini_batch`

### 문제 원문

단일 샘플에서 `outer(delta, x)`로 계산하던 gradient가 batch에서는 왜 행렬곱으로 바뀌는지 설명하시오. batch dimension은 0번째 축으로 둔다.

| 기호 | 의미 | shape |
|---|---|---:|
| X |  | `(B,D)` |
| W |  | `(H,D)` |
| Z |  |  |
| dZ |  |  |
| dW |  |  |
| db |  |  |
| dX |  |  |

다음 문장을 완성하시오.

> `Network.forward`는 layers를 ______ 순서로 돌고, `Network.backward`는 layers를 ______ 순서로 돈다. 각 layer는 전체 network를 몰라도 된다. 왜냐하면 각 layer는 위층 gradient와 자기 ________만 알면 되기 때문이다.


### 채점 기준
vectorization 설명 2점, shape 표 3점, forward/backward 순서 1점, layer 분업 1점

### 유도 질문 / 자주 틀리는 지점
batch axis를 feature axis와 섞지 말 것.


In [ ]:
# 힌트:
# arrays = np.load(DATA_DIR / "minibatch_shape_examples.npz")
# X, W, dZ의 shape를 확인하고 dW/db/dX의 shape를 추론한다.
# Network는 layer list를 보관하고 reversed(layers)를 사용할 수 있다.


### 내 답안


- outer product가 batch 행렬곱으로 바뀌는 이유:
| 기호 | 의미 | shape |
|---|---|---:|
| X |  |  |
| W |  |  |
| Z |  |  |
| dZ |  |  |
| dW |  |  |
| db |  |  |
| dX |  |  |
- 문장 완성:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 문제 16. Keras 연결과 최종 binary classification 종합 [Q16-OP, 10점]

**Source nodes:** `n_ML13.keras`, `n_ML12.network_mini_batch`

### 문제 원문

`binary_classification_toy.csv`는 작은 이진 분류 데이터다. 다음을 모두 판단하시오.

1. `X`와 `y`는 어떤 column인가?
2. input shape는 어떻게 표현하는가?
3. output layer는 어떤 형태가 적절한가?
4. activation, loss, metric은 무엇을 고를 수 있는가?
5. forward/loss/backward/update 흐름을 5줄 이내로 설명하시오.
6. mini-batch shape를 예시 `B=8`로 쓰시오.
7. validation이 왜 필요한지 설명하시오.
8. 직접 구현 `Network`와 Keras `Sequential`을 1:1 대응시키시오.
9. `compile`, `fit`, `evaluate`가 직접 구현에서 무엇에 해당하는지 설명하시오.
10. optimizer는 gradient를 계산하는가, 아니면 계산된 gradient로 parameter를 움직이는가?

| 직접 구현 | Keras | 설명 |
|---|---|---|
| Network() |  |  |
| net.add(Dense(...)) |  |  |
| ReLU() |  |  |
| SoftmaxCE 또는 BCE |  |  |
| 직접 train loop |  |  |
| 직접 test/validation 평가 |  |  |


### 채점 기준
X/y와 shape 2점, output/loss/metric 2점, 학습 흐름 2점, validation 1점, Keras 대응 2점, optimizer 책임 1점

### 유도 질문 / 자주 틀리는 지점
Keras 문법 암기보다 직접 구현 구조와의 대응을 먼저 설명할 것.


In [ ]:
# 힌트:
# binary = pd.read_csv(DATA_DIR / "binary_classification_toy.csv")
# feature column과 target column을 구분한다.
# Keras mapping은 구조 연결 확인용이며, 본 범위는 Gate 3가 아니다.
# optimizer의 책임 문장을 반드시 분리해서 쓴다.


### 내 답안


- X columns:
- y column:
- input shape:
- output layer / activation / loss / metric:
- mini-batch shape(B=8):
- validation 필요성:
| 직접 구현 | Keras | 설명 |
|---|---|---|
| Network() |  |  |
| net.add(Dense(...)) |  |  |
| ReLU() |  |  |
| loss object |  |  |
| train loop |  |  |
| evaluation |  |  |
- optimizer 책임:


### 자체 점검
- [ ] 문제에서 요구한 모든 shape를 썼다.
- [ ] `@`와 `*`를 구분했다.
- [ ] optimizer의 책임을 gradient 계산과 update로 구분했다.
- [ ] 정답본을 보기 전에 내 언어로 근거를 썼다.


## 제출 전 Self-Checklist

- [ ] 문제 1~4에서 Perceptron, XOR, loss/gradient, chain rule을 내 말로 설명했다.
- [ ] 문제 5~8에서 forward cache와 shape를 누락하지 않았다.
- [ ] 문제 9~12에서 `delta2`, `W2.T`, `delta1`, `dW1/db1`을 순서대로 연결했다.
- [ ] 문제 13~15에서 `Dense.backward`, activation backward, SoftmaxCE, mini-batch, Network abstraction을 구분했다.
- [ ] 문제 16에서 직접 구현과 Keras를 1:1로 대응시켰다.
- [ ] “Optimizer는 gradient를 계산하는 존재가 아니라, 계산된 gradient로 parameter를 update하는 규칙”이라고 설명할 수 있다.
